In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

import shapely
from shapely.geometry import Point

In [2]:
F2024_ORIG = r'Q:\CMP\LOS Monitoring 2024\Network_Conflation\v2401\CMP_Segment_INRIX_Links_Correspondence_2401_Manual-expandednetwork.csv'
F2024_CMP_NEW = r'Q:\CMP\LOS Monitoring 2024\Network_Conflation\v2401\CMP_Segment_INRIX_Links_Correspondence_2401_Manual.csv'
F2025 = r'Q:\Data\Observed\Streets\INRIX\v2501\network_conflation\CMP\CMP_Segment_INRIX_Links_Correspondence_2501_Manual.csv'
CMP = r'Q:\GIS\Transportation\Roads\CMP\cmp_roadway_segments-expanded-v202204.gpkg'
XD2024 = r'Q:\GIS\Transportation\Roads\INRIX\XD\2401\INRIX_XD-SF-2401.gpkg'

In [3]:
corr2025 = pd.read_csv(F2025)
corr2024_orig = pd.read_csv(F2024_ORIG)
cmp = gpd.read_file(CMP)
xd = gpd.read_file(XD2024)

In [4]:
add = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2501\maprelease-xdadded\USA_California.csv')
rep = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2501\maprelease-xdreplaced\USA_California.csv')
rem = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2501\maprelease-xdremoved\USA_California.csv')
#rv2501.rename(columns={'XDId_25_1':'new','XDId_24_1':'old'}, inplace=True)
#rv2501['version'] = 2501

In [5]:
rep

,XDId_25_1,XDId_24_1,StartOffset,EndOffset
0,937055443,429487217,0.00,35.58
1,1626717916,429487217,35.58,41.14
2,937050709,476179109,0.00,12.10
3,937017183,485590968,0.00,4.22
4,937014371,485570490,4.20,17.54
...,...,...,...,...
15507,937118415,449944941,66.98,231.78
15508,172652421,449944941,0.00,28.70
15509,937162804,172998515,129.33,325.20
15510,937147111,172998515,325.20,407.20


In [6]:
corr2025.loc[corr2025['INRIX_SegID'].isin(rep['XDId_25_1'])]

,CMP_SegID,INRIX_SegID,Length_Matched
239,23,937045027,1646.820803
240,23,937045027,1641.814494
262,27,937040647,213.565096
265,27,1626734566,994.051005
283,30,937077505,1606.528057
614,68,937010064,307.132395
615,68,937075397,68.473867
623,69,937009803,302.394016
624,69,937071610,83.533901
675,77,937044879,473.568478


In [7]:
corr2024_orig.loc[corr2024_orig['INRIX_SegID'].isin(rem['SegId'])]

,CMP_SegID,INRIX_SegID,Length_Matched
698,86,485563122,487.430384
709,87,476173725,730.362231
4550,738,170108638,456.054698


In [8]:
corr2025.loc[corr2025['INRIX_SegID'].isin(add['SegId'])]

,CMP_SegID,INRIX_SegID,Length_Matched
239,23,937045027,1646.820803
240,23,937045027,1641.814494
262,27,937040647,213.565096
283,30,937077505,1606.528057
614,68,937010064,307.132395
615,68,937075397,68.473867
623,69,937009803,302.394016
624,69,937071610,83.533901
675,77,937044879,473.568478
699,82,937047777,425.535368


In [9]:
tmp = pd.merge(
    corr2025,
    rep[['XDId_25_1', 'XDId_24_1']],
    left_on='INRIX_SegID',
    right_on='XDId_25_1',
    how='left'
)

In [10]:
def get_matched_length(cmp, xd, cmp_segid, xd_segid):
    ls1 = cmp.loc[cmp['cmp_segid'].eq(cmp_segid)].to_crs('epsg:2227').iloc[0]['geometry'].geoms[0]
    ls2 = xd.loc[xd['XDSegID'].eq(xd_segid)].to_crs('epsg:2227').iloc[0]['geometry']
    a1 = Point(ls1.coords[0])
    b1 = Point(ls1.coords[-1])
    return abs(ls2.project(a1)-ls2.project(b1))

In [11]:
for idx, row in tmp.dropna().iterrows():
    if row['XDId_24_1'] in xd['XDSegID'].tolist():
        tmp.loc[idx, 'Length_Matched'] = get_matched_length(cmp, xd, row['CMP_SegID'], row['XDId_24_1'])
        tmp.loc[idx, 'INRIX_SegID'] = row['XDId_24_1']
    else:
        print('skipping update, XDSegID not found index: {}, cmp: {}, xdsegid: {}'.format(idx, row['CMP_SegID'], row['XDId_24_1']))

skipping update, XDSegID not found index: 713, cmp: 82.0, xdsegid: 476177631.0


In [12]:
manual_add = [
    (86, 485563122),
    (81, 1626683690),
    (87, 476173725), 
    (100, 400410149),
    (165, 1626698075),
    (166, 1626698075),
    (170, 441577189),
    (234, 1626628033),
    (238, 1626710692),
]

In [13]:
recs = []
for cmp_segid, xd_segid in manual_add:
    recs.append([cmp_segid, xd_segid, get_matched_length(cmp, xd, cmp_segid, xd_segid)])

In [14]:
corr2024 = pd.concat(
    [tmp[['CMP_SegID','INRIX_SegID','Length_Matched']].drop_duplicates(subset=['CMP_SegID','INRIX_SegID']),
    pd.DataFrame(recs, columns=['CMP_SegID','INRIX_SegID','Length_Matched'])]
).sort_values(['CMP_SegID']).reset_index(drop='True')

In [15]:
corr2024

,CMP_SegID,INRIX_SegID,Length_Matched
0,1,449830550,626.443004
1,1,449830551,637.111451
2,1,449851041,633.909643
3,1,449851042,219.825607
4,1,450493283,188.741166
...,...,...,...
1976,245,400070922,149.289919
1977,245,1626728989,2145.930504
1978,245,1626729010,2145.936884
1979,245,429464987,2145.956925


In [16]:
corr2024.to_csv(F2024_CMP_NEW, index=False)

In [17]:
corr2025.loc[corr2025['INRIX_SegID'].eq(485563122)]

,CMP_SegID,INRIX_SegID,Length_Matched
